#Configuration

In [0]:
CATALOG = "cms_medicare_databricks_pipeline"
SOURCE_TABLE = f"{CATALOG}.bronze.provider_utilization_raw"
TARGET_PROVIDERS = f"{CATALOG}.silver.providers"
TARGET_PROCEDURES = f"{CATALOG}.silver.procedures"

In [0]:
df_bronze = spark.read.table(SOURCE_TABLE)

print(f"Bronze row count: {df_bronze.count():,}")
print(f"Bronze columns: {len(df_bronze.columns)}")


#Filter

In [0]:
from pyspark.sql import functions as F

df_california = df_bronze.filter(
    F.col("Rndrng_Prvdr_State_Abrvtn") == "CA"
)

ca_count = df_california.count()
print(f"California row count: {ca_count:,}")
print(f"Reduction: {(1 - ca_count / df_bronze.count()) * 100:.2f}% of national data filtered out")

#Providers Table

In [0]:
df_providers = df_california.select(
    # --- Provider identity ---
    F.col("Rndrng_NPI").alias("provider_npi"),
    F.col("Rndrng_Prvdr_Last_Org_Name").alias("provider_last_name"),
    F.col("Rndrng_Prvdr_First_Name").alias("provider_first_name"),
    F.col("Rndrng_Prvdr_Crdntls").alias("credentials"),
    F.col("Rndrng_Prvdr_Ent_Cd").alias("entity_code"),
    # Entity code: I = Individual provider, O = Organization

    # --- Location ---
    F.col("Rndrng_Prvdr_City").alias("city"),
    F.col("Rndrng_Prvdr_State_Abrvtn").alias("state"),
    F.col("Rndrng_Prvdr_Zip5").alias("zip_code"),
    F.col("Rndrng_Prvdr_RUCA").cast("decimal(5,2)").alias("ruca_code"),
    # RUCA = Rural-Urban Commuting Area code

    # --- Provider type ---
    F.col("Rndrng_Prvdr_Type").alias("provider_type"),
    F.col("Rndrng_Prvdr_Mdcr_Prtcptg_Ind").alias("medicare_participating"),

    # --- Metadata from bronze table ---
    F.col("ingestion_timestamp"),
    F.col("source_file")
).dropDuplicates(["provider_npi"])
# dropDuplicates on NPI(National Provider Identifier) because the same provider appears

print(f"Unique California providers: {df_providers.count():,}")

#Procedures Table

In [0]:
df_procedures = df_california.select(
    F.col("Rndrng_NPI").alias("provider_npi"),
    # --- Procedure identity ---
    F.col("HCPCS_Cd").alias("hcpcs_code"),
    # HCPCS = Healthcare Common Procedure Coding System
    F.col("HCPCS_Desc").alias("hcpcs_description"),
    F.col("HCPCS_Drug_Ind").alias("drug_indicator"),
    F.col("Place_Of_Srvc").alias("place_of_service"),
    # --- Volume columns ---
    F.expr("try_cast(Tot_Benes as decimal(12,2))").alias("total_beneficiaries"),
    F.expr("try_cast(Tot_Srvcs as decimal(12,2))").alias("total_services"),
    F.expr("try_cast(Tot_Bene_Day_Srvcs as decimal(12,2))").alias("total_beneficiary_day_services"),
    # --- Cost columns ---
    F.expr("try_cast(Avg_Sbmtd_Chrg as decimal(12,2))").alias("avg_submitted_charge"),
    F.expr("try_cast(Avg_Mdcr_Alowd_Amt as decimal(12,2))").alias("avg_medicare_allowed_amount"),
    F.expr("try_cast(Avg_Mdcr_Pymt_Amt as decimal(12,2))").alias("avg_medicare_payment_amount"),
    F.expr("try_cast(Avg_Mdcr_Stdzd_Amt as decimal(12,2))").alias("avg_medicare_standardized_amount"),
    # --- Audit trail ---
    F.col("ingestion_timestamp"),
    F.col("source_file")
).dropDuplicates(["provider_npi", "hcpcs_code", "place_of_service"])

#Data Quality Checks

In [0]:
# Count violations
null_npi = df_procedures.filter(
    F.col("provider_npi").isNull()
    ).count()

invalid_payment = df_procedures.filter(
    F.col("avg_medicare_payment_amount") <= 0
    # Payment of zero or negative makes no business sense
).count()

null_hcpcs = df_procedures.filter(
    F.col("hcpcs_code").isNull()
    # A procedure row with no procedure code is meaningless
).count()

# Print quality report
print("=== DATA QUALITY REPORT ===")
print(f"Null NPI records:          {null_npi:,}")
print(f"Invalid payment records:   {invalid_payment:,}")
print(f"Null HCPCS code records:   {null_hcpcs:,}")
print(f"Total procedures:          {df_procedures.count():,}")

df_procedures_clean = df_procedures.filter(
    F.col("provider_npi").isNotNull() &
    (F.col("avg_medicare_payment_amount") > 0) &
    F.col("hcpcs_code").isNotNull()
)

df_quarantine = df_procedures.filter(
    F.col("provider_npi").isNull() |
    (F.col("avg_medicare_payment_amount") <= 0) |
    F.col("hcpcs_code").isNull()
)

print(f"\nClean records:             {df_procedures_clean.count():,}")
print(f"Quarantined records:       {df_quarantine.count():,}")

#Silver Table

In [0]:
# Write to silver unique california providers table
(
    df_providers
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_PROVIDERS)
)
print(f"silver.providers written: {df_providers.count():,} rows")

# Write clean procedures only
(
    df_procedures_clean
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_PROCEDURES)
)
print(f"silver.procedures written: {df_procedures.count():,} rows")

# Write quarantine records
(
    df_quarantine
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.silver.procedures_quarantine")
)
print(f"silver.procedures_quarantine written: {df_quarantine.count():,} rows")

#Verification

In [0]:
providers_count = spark.read.table(TARGET_PROVIDERS).count()
procedures_count = spark.read.table(TARGET_PROCEDURES).count()
quarantine_count = spark.read.table(
    f"{CATALOG}.silver.procedures_quarantine"
).count()

print("=== SILVER LAYER SUMMARY ===")
print(f"silver.providers:             {providers_count:,} unique CA providers")
print(f"silver.procedures:            {procedures_count:,} clean procedure rows")
print(f"silver.procedures_quarantine: {quarantine_count:,} quarantined rows")

print("\n--- Sample providers ---")
display(spark.read.table(TARGET_PROVIDERS).limit(5))

print("\n--- Sample procedures ---")
display(spark.read.table(TARGET_PROCEDURES).limit(5))